In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pour avoir des graphiques propres directement dans le notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

### **(0). Loading & Cleaning Data**

Dataset :
- prices.csv 'price_da' 
- Indicative Forecasts by Zone
- Indicative Generation (DA & ID)
- Indicative Margin (DA & ID)
- Transmission Demand by Zone (DA & ID) 


In [ ]:
# Load target file
# price_DA -> what we know
# price_ID -> what we want to forecast

prices = pd.read_csv('dataset/prices.csv', parse_dates=['GMT_PERIOD_START_DATETIME']) # parse_dates convert date column to datetime
prices = prices.rename(columns={'GMT_PERIOD_START_DATETIME': 'timestamp'})
prices = prices.sort_values('timestamp').reset_index(drop=True)

print(prices.shape)
prices.head()

(87696, 3)


,timestamp,price_DA,price_ID
0,2020-01-01 00:00:00,43.45,35.28
1,2020-01-01 00:30:00,43.45,37.01
2,2020-01-01 01:00:00,41.12,37.78
3,2020-01-01 01:30:00,41.12,35.60
4,2020-01-01 02:00:00,29.98,31.70


In [ ]:
# load features file

feature_files = {
    'indgen_da': 'dataset/INDGEN_DA.csv',
    'indgen_id': 'dataset/INDGEN_ID.csv',
    'melngc_da': 'dataset/MELNGC_DA.csv',
    'melngc_id': 'dataset/MELNGC_ID.csv',
    'tsdf_da':   'dataset/TSDF_DA.csv',
    'tsdf_id':   'dataset/TSDF_ID.csv',
}

dfs = {}
for name, path in feature_files.items():

    df = pd.read_csv(path, parse_dates=['GMT_PERIOD_START_DATETIME'])
    df = df.rename(columns={'GMT_PERIOD_START_DATETIME': 'timestamp'})
    df = df.sort_values('timestamp').reset_index(drop=True)

    # Prefix zone columns to avoid name conflicts when merging: Z1 → indgen_da_Z1
    zone_cols = [c for c in df.columns if c != 'timestamp']
    df = df.rename(columns={z: f'{name}_{z}' for z in zone_cols})
    dfs[name] = df

# Quick check
for name, df in dfs.items():
    print(f"{name}: {df.shape} | {df['timestamp'].min()} → {df['timestamp'].max()}")

indgen_da: (43848, 18) | 2020-01-01 00:00:00 → 2024-12-31 23:00:00
indgen_id: (43848, 18) | 2020-01-01 00:00:00 → 2024-12-31 23:00:00
melngc_da: (43848, 18) | 2020-01-01 00:00:00 → 2024-12-31 23:00:00
melngc_id: (43848, 18) | 2020-01-01 00:00:00 → 2024-12-31 23:00:00
tsdf_da: (43848, 18) | 2020-01-01 00:00:00 → 2024-12-31 23:00:00
tsdf_id: (43848, 18) | 2020-01-01 00:00:00 → 2024-12-31 23:00:00
